# HOTOSM Highway Segmentation Starter Notebook

Train a small U-Net from scratch on aerial tiles and export road centerlines in the challenge submission format.

1. Draw the labeled centerlines as thin training masks.
2. Train the model and choose a probability threshold on labeled validation tiles.
3. Thin the predicted masks to centerlines, trace them as lines, and write `submission.csv` with the three required columns. Leave `metres_per_pixel` blank.

Extract the challenge data and point `DATA_DIR` at the folder containing `Train.csv`, `Test.csv`, `SampleSubmission.csv`, and `images/<ID>.png`. A GPU is recommended for training. In Colab, enable a GPU under **Runtime → Change runtime type** and set the path to your extracted files. Use the split supplied by the organizers if available.

If imports are missing, run `%pip install torch numpy scipy shapely pillow` in a notebook cell and restart the kernel if prompted.

In [ ]:
import csv
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from scipy.ndimage import label
from shapely import from_wkt, get_parts, to_wkt
from shapely.geometry import LineString, MultiLineString
from torch import nn
from torch.utils.data import DataLoader, Dataset

DATA_DIR = Path("zindi")  # Folder with the extracted challenge data
OUTPUT_CSV = Path("submission.csv")
SEED = 42
MODEL_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 0.001

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
if device.type == "cpu":
    torch.set_num_threads(min(8, os.cpu_count() or 1))
print(f"Device: {device}")

## 1. Load the data and draw training masks

Each row in `Train.csv` holds one tile and its road centerlines in WKT. Coordinates are in the tile's native pixel grid. Draw a 5-pixel-wide band around the lines at native resolution and resize the mask for training. Empty geometries give an all-background mask.

Use `Val.csv` when supplied, or the `val` rows in `id_map.csv` when available. Otherwise the notebook reserves a seeded sample from the labeled rows for a quick check. Nearby tiles can appear in both subsets with that fallback, so use a geographic split when measuring generalization.

In [ ]:
def read_csv(filename, required_columns):
    with (DATA_DIR / filename).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not set(required_columns).issubset(reader.fieldnames or []):
            raise ValueError(f"{filename} needs columns {required_columns}")
        rows = list(reader)
    ids = [row["ID"] for row in rows]
    if not ids or any(not tile_id for tile_id in ids) or len(ids) != len(set(ids)):
        raise ValueError(f"{filename} needs one row per unique, nonempty ID")
    return rows


labeled = read_csv("Train.csv", ("ID", "geometry"))
sample = read_csv("SampleSubmission.csv", ("ID", "geometry"))
test_ids = {row["ID"] for row in read_csv("Test.csv", ("ID",))}
sample_ids = [row["ID"] for row in sample]
if set(sample_ids) != test_ids:
    raise ValueError("Test.csv and SampleSubmission.csv must contain the same IDs")

if (DATA_DIR / "Val.csv").is_file():
    train_rows = labeled
    val_rows = read_csv("Val.csv", ("ID", "geometry"))
    split_name = "Val.csv"
else:
    val_ids = set()
    if (DATA_DIR / "id_map.csv").is_file():
        split_rows = read_csv("id_map.csv", ("ID", "split"))
        val_ids = {row["ID"] for row in split_rows if row["split"] == "val"}
    if val_ids.intersection(row["ID"] for row in labeled):
        split_name = "id_map.csv"
    else:
        shuffled = labeled.copy()
        random.Random(SEED).shuffle(shuffled)
        val_ids = {row["ID"] for row in shuffled[:max(1, len(labeled) // 10)]}
        split_name = "seeded labeled holdout"
    train_rows = [row for row in labeled if row["ID"] not in val_ids]
    val_rows = [row for row in labeled if row["ID"] in val_ids]

train_ids = {row["ID"] for row in train_rows}
val_ids = {row["ID"] for row in val_rows}
if not train_ids or not val_ids or train_ids & val_ids or (train_ids | val_ids) & test_ids:
    raise ValueError("Training, validation and test must be nonempty and have disjoint IDs")

all_ids = train_ids | val_ids | test_ids
missing = sorted(tile_id for tile_id in all_ids if not (DATA_DIR / "images" / f"{tile_id}.png").is_file())
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} tile images, including {missing[:3]}")
print(f"{len(train_ids):,} train, {len(val_ids):,} validation ({split_name}), {len(test_ids):,} test tiles")

In [ ]:
class RoadTiles(Dataset):
    def __init__(self, rows, labeled=False):
        self.rows = rows
        self.labeled = labeled

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        tile_id = row["ID"]
        with Image.open(DATA_DIR / "images" / f"{tile_id}.png") as source:
            width, height = source.size
            resized = source.convert("RGB").resize(
                (MODEL_SIZE, MODEL_SIZE), Image.Resampling.BILINEAR
            )
        image = torch.from_numpy(np.array(resized)).permute(2, 0, 1).float() / 255
        if not self.labeled:
            return image, tile_id, width, height

        mask = Image.new("L", (width, height))
        draw = ImageDraw.Draw(mask)
        for line in get_parts(from_wkt(row["geometry"])):
            if line.geom_type == "LineString" and line.length > 0:
                draw.line(list(line.coords), fill=1, width=5)
        target = torch.from_numpy(np.array(mask.resize(
            (MODEL_SIZE, MODEL_SIZE), Image.Resampling.NEAREST
        ))).float().unsqueeze(0)
        return image, target


loader_options = {"batch_size": BATCH_SIZE, "pin_memory": device.type == "cuda"}
train_loader = DataLoader(RoadTiles(train_rows, labeled=True), shuffle=True, **loader_options)
val_loader = DataLoader(RoadTiles(val_rows, labeled=True), **loader_options)
test_loader = DataLoader(RoadTiles(sample), **loader_options)

## 2. Train a small U-Net

The encoder downsamples the tile twice. The decoder upsamples and reuses encoder features to preserve spatial detail. The loss combines binary cross-entropy and Dice loss because road pixels cover a small part of each tile.

Choose the checkpoint by validation loss and select the threshold by validation **mask F1**, summed over validation pixels. This quick training diagnostic measures the masks. The competition scorer evaluates the exported centerlines and road graph.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder1 = conv_block(3, 16)
        self.encoder2 = conv_block(16, 32)
        self.bottleneck = conv_block(32, 64)
        self.decoder2 = conv_block(64 + 32, 32)
        self.decoder1 = conv_block(32 + 16, 16)
        self.output = nn.Conv2d(16, 1, 1)

    def forward(self, images):
        first = self.encoder1(images)
        second = self.encoder2(F.max_pool2d(first, 2))
        center = self.bottleneck(F.max_pool2d(second, 2))
        up = F.interpolate(center, size=second.shape[-2:], mode="bilinear", align_corners=False)
        up = self.decoder2(torch.cat((up, second), dim=1))
        up = F.interpolate(up, size=first.shape[-2:], mode="bilinear", align_corners=False)
        return self.output(self.decoder1(torch.cat((up, first), dim=1)))


def segmentation_loss(logits, targets):
    probabilities = logits.sigmoid()
    axes = (1, 2, 3)
    overlap = (probabilities * targets).sum(dim=axes)
    total = probabilities.sum(dim=axes) + targets.sum(dim=axes)
    dice = (2 * overlap + 1) / (total + 1)
    return F.binary_cross_entropy_with_logits(logits, targets) + 1 - dice.mean()


model = TinyUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(f"{sum(p.numel() for p in model.parameters()):,} model parameters")

In [ ]:
@torch.inference_mode()
def validation_loss():
    model.eval()
    total = 0.0
    for images, masks in val_loader:
        loss = segmentation_loss(model(images.to(device)), masks.to(device))
        total += loss.item() * len(images)
    return total / len(val_rows)


best_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, masks in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = segmentation_loss(model(images.to(device)), masks.to(device))
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(images)

    val_loss = validation_loss()
    if val_loss < best_loss:
        best_loss = val_loss
        best_weights = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    print(f"Epoch {epoch:02d} | train loss {running_loss / len(train_rows):.4f} | val loss {val_loss:.4f}")

model.load_state_dict(best_weights)
model.eval()

In [ ]:
@torch.inference_mode()
def validation_threshold(candidates):
    matched = dict.fromkeys(candidates, 0)
    predicted = dict.fromkeys(candidates, 0)
    labeled_pixels = 0
    for images, masks in val_loader:
        probabilities = model(images.to(device)).sigmoid().cpu().numpy()
        truth = masks.numpy() > 0
        labeled_pixels += int(truth.sum())
        for threshold in candidates:
            guess = probabilities >= threshold
            matched[threshold] += int((guess & truth).sum())
            predicted[threshold] += int(guess.sum())
    if not labeled_pixels:
        raise ValueError("Validation needs some tiles with labeled roads")
    scores = {
        t: 2 * matched[t] / (predicted[t] + labeled_pixels)
        for t in candidates
    }
    best = max(scores, key=scores.get)
    return best, scores


threshold, validation_f1 = validation_threshold((0.15, 0.25, 0.35, 0.45, 0.55))
for value, score in validation_f1.items():
    print(f"Threshold {value:.2f} | validation mask F1 {score:.4f}")
print(f"Using threshold {threshold:.2f}")

## 3. Export predicted road centerlines

Resize each probability map back to the source tile, apply the selected threshold, and thin the binary mask to a one-pixel skeleton. Trace segments between junctions and endpoints and simplify each polyline. Short isolated components are dropped to reduce noise.

The output contains exactly the template's IDs and row order. Coordinates are `(x, y)` in native tile pixels, with `x` pointing right and `y` down from the top-left. Each `geometry` is a WKT `MULTILINESTRING`, or `MULTILINESTRING EMPTY` for a tile without predicted roads. The CSV writer quotes fields that contain commas.

In [ ]:
def thin_mask(binary):
    """Thin a padded binary mask with the Zhang–Suen algorithm."""
    mask = (binary > 0).astype(np.uint8)
    center = mask[1:-1, 1:-1]
    neighbors = (mask[:-2, 1:-1], mask[:-2, 2:], mask[1:-1, 2:], mask[2:, 2:],
                 mask[2:, 1:-1], mask[2:, :-2], mask[1:-1, :-2], mask[:-2, :-2])
    north, _, east, _, south, _, west, _ = neighbors
    while True:
        changed = False
        for a, b, c, d in ((north, east, south, west), (east, north, west, south)):
            count = sum(neighbors)
            transitions = sum((n == 0).astype(np.uint8) & following
                              for n, following in zip(neighbors, neighbors[1:] + neighbors[:1]))
            delete = ((center != 0) & (count >= 2) & (count <= 6) & (transitions == 1)
                      & (a * b * c == 0) & (b * c * d == 0))
            changed |= bool(delete.any())
            center[delete] = 0
        if not changed:
            return mask * 255


def paths(skeleton):
    """Walk each skeleton edge once, including closed loops."""
    pixels = {tuple(p) for p in np.argwhere(skeleton)}
    graph = {}
    for row, col in sorted(pixels):
        adjacent = []
        for dy, dx in ((-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)):
            other = row + dy, col + dx
            if other in pixels and not (dy and dx and ((row + dy, col) in pixels or (row, col + dx) in pixels)):
                adjacent.append(other)
        graph[row, col] = adjacent
    used = set()
    for node in sorted(graph, key=lambda item: (len(graph[item]) == 2, item)):
        for neighbour in graph[node]:
            edge = tuple(sorted((node, neighbour)))
            if edge in used:
                continue
            result = [node, neighbour]
            used.add(edge)
            while len(graph[result[-1]]) == 2:
                a, b = graph[result[-1]]
                following = b if a == result[-2] else a
                edge = tuple(sorted((result[-1], following)))
                if edge in used:
                    break
                used.add(edge)
                result.append(following)
            yield result

In [ ]:
MIN_COMPONENT_PIXELS = 12
MAX_VERTICES = 4096


def predict_geometry(probability, width, height):
    resized = np.asarray(Image.fromarray(probability).resize(
        (width, height), Image.Resampling.BILINEAR
    ))
    skeleton = thin_mask(np.pad(resized >= threshold, 1))[1:-1, 1:-1] > 0
    components, _ = label(skeleton, structure=np.ones((3, 3), dtype=bool))
    keep = np.bincount(components.ravel()) >= MIN_COMPONENT_PIXELS
    keep[0] = False

    lines = []
    for trace in paths(keep[components]):
        line = LineString([(col + 0.5, row + 0.5) for row, col in trace]).simplify(0.6)
        if line.length > 0:
            lines.append(line)

    if sum(len(line.coords) for line in lines) > MAX_VERTICES:
        # Keep the longest segments first if a noisy mask exceeds the scoring limit.
        kept, remaining = [], MAX_VERTICES
        for line in sorted(lines, key=lambda line: line.length, reverse=True):
            vertices = len(line.coords)
            if vertices <= remaining:
                kept.append(line)
                remaining -= vertices
        lines = kept
    return MultiLineString(lines)


@torch.inference_mode()
def predict_test():
    model.eval()
    for images, ids, widths, heights in test_loader:
        probabilities = model(images.to(device)).sigmoid().cpu().numpy()[:, 0]
        for probability, tile_id, width, height in zip(probabilities, ids, widths, heights):
            width, height = int(width), int(height)
            geometry = predict_geometry(probability, width, height)
            yield tile_id, to_wkt(geometry, rounding_precision=3), width, height


predictions = list(predict_test())
if [row[0] for row in predictions] != sample_ids:
    raise RuntimeError("Prediction IDs or order differ from SampleSubmission.csv")

nonempty = 0
for tile_id, wkt, width, height in predictions:
    geometry = from_wkt(wkt)
    if geometry.geom_type != "MultiLineString" or not geometry.is_valid:
        raise ValueError(f"Invalid centerlines for {tile_id}")
    if not geometry.is_empty:
        nonempty += 1
        x_min, y_min, x_max, y_max = geometry.bounds
        if not (0 <= x_min <= x_max <= width and 0 <= y_min <= y_max <= height):
            raise ValueError(f"Centerlines outside tile bounds for {tile_id}")

with OUTPUT_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["ID", "geometry", "metres_per_pixel"])
    writer.writerows((tile_id, wkt, "") for tile_id, wkt, _, _ in predictions)

print(f"Wrote {len(predictions):,} rows to {OUTPUT_CSV.resolve()}")
print(f"Tiles with predicted roads: {nonempty:,}")
if nonempty == 0:
    print("All tiles are empty. Check validation F1 and train for more epochs before submitting.")

## 4. Check a prediction

The cyan lines are the geometry that went into the CSV. In Colab, download `submission.csv` from the Files sidebar.

In [ ]:
from IPython.display import display

example = next((row for row in predictions if row[1] != "MULTILINESTRING EMPTY"), predictions[0])
tile_id, wkt, _, _ = example
with Image.open(DATA_DIR / "images" / f"{tile_id}.png") as image:
    preview = image.convert("RGB")
draw = ImageDraw.Draw(preview)
for line in get_parts(from_wkt(wkt)):
    if not line.is_empty:
        draw.line(list(line.coords), fill="#00ffff", width=2)
print(tile_id)
display(preview)